# 文本相似度实例

## Step1 导入相关包

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification,Trainer,TrainingArguments
from datasets import load_dataset

## Step2 加载数据集

In [ ]:
dataset = load_dataset("json", data_files="./train_pair_1w.json", split="train")
dataset

In [ ]:
dataset[0]

## Step3 划分数据集

In [ ]:
datasets = dataset.train_test_split(test_size=0.2)
datasets

## Step4 数据预处理

In [ ]:
import torch

tokenizer = AutoTokenizer.from_pretrained("hfl/chinese-macbert-base")

def process_function(examples):
    tokenized_examples = tokenizer(examples["sentence1"], examples["sentence2"], max_length=128, truncation=True)  # ty:ignore[call-non-callable]
    # 每对数据输出一个score，用均方误差做loss，标签需要是float类型
    tokenized_examples["labels"] = [float(label) for label in examples["label"]]
    return tokenized_examples

tokenized_datasets = datasets.map(process_function, batched=True, remove_columns=datasets["train"].column_names)
tokenized_datasets

In [ ]:
print(tokenized_datasets["train"][0])

## Step5 创建模型

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained("hfl/chinese-macbert-base",num_labels = 1)

In [ ]:
from transformers import BertForSequenceClassification

## Step6 创建评估函数

In [ ]:
import evaluate

acc_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

In [ ]:
def eval_metric(eval_predict):
    predicts, labels = eval_predict
    predicts = [int(p > 0.5) for p in predicts]
    labels = [int(l) for l in labels]
    acc = acc_metric.compute(predictions=predicts, references=labels) or {}  # ty:ignore[missing-argument]
    f1 = f1_metric.compute(predictions=predicts, references=labels) or {}  # ty:ignore[missing-argument]
    acc.update(f1)  
    return acc

## Step7 创建TrainingArguments

In [ ]:
train_args = TrainingArguments(
    output_dir="./cross_model",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=128,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    metric_for_best_model="f1",
    load_best_model_at_end=True,
)
train_args

## Step8 创建Trainer

In [ ]:
from transformers import DataCollatorWithPadding

trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=eval_metric,
)

## Step9 模型训练

In [ ]:
trainer.train()

## Step10 模型评估


In [ ]:
trainer.evaluate(tokenized_datasets["test"])

## Step11 模型预测

In [ ]:
trainer.predict(tokenized_datasets["test"])

In [ ]:
from transformers import pipeline

In [ ]:
model.config.id2label = {0: "不相似", 1: "相似"}

In [ ]:
pipe = pipeline("text-classification",model=model, tokenizer=tokenizer,device=0)

In [ ]:
result = pipe({"text": "我喜欢吃苹果", "text_pair": "我喜欢吃香蕉"},function_to_apply="sigmoid")
result["label"] = "相似" if result["score"] > 0.5 else "不相似"
print(result)